In [0]:
from pyspark.sql import functions as F
from datetime import date

# -----------------------------------------------------------------------------
# Recupera a menor e maior data da fato
# -----------------------------------------------------------------------------

periodo = (
    spark.table("dev_credit_risk.silver.fact_contrato")
         .select(
             F.min("dt_contratacao").alias("dt_inicio"),
             F.max("dt_contratacao").alias("dt_fim")
         )
         .first()
)

# -----------------------------------------------------------------------------
# Obtém apenas os anos
# -----------------------------------------------------------------------------

ano_inicio = periodo["dt_inicio"].year
ano_fim = periodo["dt_fim"].year

# -----------------------------------------------------------------------------
# Define o período completo
# -----------------------------------------------------------------------------

dt_inicio = date(ano_inicio, 1, 1)
dt_fim = date(ano_fim, 12, 31)

# -----------------------------------------------------------------------------
# Gera todas as datas do período
# -----------------------------------------------------------------------------

df_datas = spark.sql(f"""
SELECT explode(
    sequence(
        to_date('{dt_inicio}'),
        to_date('{dt_fim}'),
        interval 1 day
    )
) AS data
""")

# -----------------------------------------------------------------------------
# Construção da dimensão calendário
# -----------------------------------------------------------------------------

dim_tempo = (
    df_datas
    .withColumn(
        "data_key",
        F.date_format("data", "yyyyMMdd").cast("int")
    )
    .withColumn("ano", F.year("data"))
    .withColumn(
        "semestre",
        F.when(F.month("data") <= 6, 1).otherwise(2)
    )
    .withColumn("trimestre", F.quarter("data"))
    .withColumn(
        "bimestre",
        F.ceil(F.month("data") / F.lit(2))
    )
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", F.date_format("data", "MMMM"))
    .withColumn("nome_mes_abrev", F.date_format("data", "MMM"))
    .withColumn("semana_ano", F.weekofyear("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana", F.dayofweek("data"))
    .withColumn("nome_dia", F.date_format("data", "EEEE"))
    .withColumn(
        "fim_semana",
        F.dayofweek("data").isin(1, 7)
    )
    .withColumn(
        "ultimo_dia_mes",
        F.col("data") == F.last_day("data")
    )
)

# -----------------------------------------------------------------------------
# Ordenação das colunas
# -----------------------------------------------------------------------------

dim_tempo = dim_tempo.select(
    "data_key",
    "data",
    "ano",
    "semestre",
    "trimestre",
    "bimestre",
    "mes",
    "nome_mes",
    "nome_mes_abrev",
    "semana_ano",
    "dia",
    "dia_semana",
    "nome_dia",
    "fim_semana",
    "ultimo_dia_mes"
)

# -----------------------------------------------------------------------------
# Persistência
# -----------------------------------------------------------------------------

def salvar_tabela(df, tabela):
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(tabela)
    )

salvar_tabela(dim_tempo, "dev_credit_risk.gold.dim_calendario")

In [0]:
%sql
select * from dev_credit_risk.gold.dim_tempo